# Fine-tuning with Unsloth

**mistral-7b-v0.3-bnb-4bit**


In [1]:
import os
import sys
import json
import time
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from trl import apply_chat_template
from unsloth.chat_templates import get_chat_template
import json, shutil, os
max_seq_length = 2048
dtype = None
load_in_4bit = True

os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

/home/danielhenel/Desktop/jarvis/research/text_to_command/Fine-tuning/.test_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Prepare train and test subsets

Dataset: Request commands dataset v1.0 // all supported request types

Train 80 - Test 20 split

In [2]:
def get_train_data(tokenizer):
    with open("../data/request_commands_train_dataset_v1.0.json", 'r') as file:
        raw_train_dataset = json.load(file)

    train_data = []
    for command_class in raw_train_dataset:
        for input, output in raw_train_dataset[command_class].items():
            if input and output:
                train_data.append({
        "prompt": [{"role": "user", "content": input}],
        "completion": [{"role": "assistant", "content": output}]
    })

    train_data = Dataset.from_list(train_data)
    train_data = train_data.map(apply_chat_template, fn_kwargs={"tokenizer": tokenizer})
    return train_data


def get_test_data(tokenizer):
    with open("../data/request_commands_test_dataset_v1.0.json", 'r') as file:
        raw_test_dataset = json.load(file)
    test_data = []
    for command_class in raw_test_dataset:
        for input, output in raw_test_dataset[command_class].items():
            if input and output:
                test_data.append({
        "prompt": [{"role": "user", "content": input}],
        "completion": [{"role": "assistant", "content": output}]
    })

    test_data = Dataset.from_list(test_data)
    test_data = test_data.map(apply_chat_template, fn_kwargs={"tokenizer": tokenizer})
    return test_data


def formatting_func(batch):
    texts = [p + c for p, c in zip(batch["prompt"], batch["completion"])]
    return texts

### Model, tokenizer and trainer configuration

In [3]:
def get_model_and_tokenizer(model_name) -> tuple:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name =  model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit
    )

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="mistral",
        mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
        map_eos_token=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = True,
        random_state = 3407,
    )
    return (model, tokenizer)

def get_training_args(output_dir: str) -> SFTConfig:
    args = SFTConfig(
        output_dir=output_dir,
        max_length=max_seq_length,
        completion_only_loss=False,
    )
    return args

### Training and evaluation

In [4]:
results = {}

def prompt_model(model, tokenizer, prompt):
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    input_ids = inputs["input_ids"]
    generated = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
    )
    
    new_tokens = generated[0, input_ids.shape[-1]:]
    new_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return new_text

def benchmark_model(model, tokenizer):
    test_dataset = get_test_data(tokenizer)
    predictions = {
        'Classification Result': [],
        'Prompt processing time': []
    }

    for e in test_dataset:
        prompt = e["prompt"]
        completion = e["completion"]

        start_time = time.time() 
        new_text = prompt_model(model, tokenizer, prompt)
        end_time = time.time()
        prompt_processing_time = end_time - start_time

        if new_text == completion.split("</s>")[0].strip():
            predictions['Classification Result'].append("CLASSIFICATION_CORRECT")
        else:
            predictions['Classification Result'].append("CLASSIFICATION_INCORRECT")
        predictions['Prompt processing time'].append(prompt_processing_time)

    predictions = pd.DataFrame(predictions)
    accuracy = predictions[predictions['Classification Result'] == "CLASSIFICATION_CORRECT"].shape[0] / predictions.shape[0] 
    accuracy *= 100 # as percentage
    avg_processing_time = predictions['Prompt processing time'].mean()

    return (accuracy, avg_processing_time)

In [5]:
model_name = "./models/mistral-7b-v0.3-bnb-4bit"
run_name = f"mistral_7b_finetuned"
output_dir = f"outputs/{run_name}"
print(f"\nRunning config for {model_name}: {run_name}")

# Get model and tokenizer
model, tokenizer = get_model_and_tokenizer(model_name)

# Get train, test and validation data
train_dataset = get_train_data(tokenizer) 

args = get_training_args(output_dir)
# Train and evaluate
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    args = args,
    train_dataset=train_dataset,
    formatting_func=formatting_func
)
trainer.train()


Running config for ./models/mistral-7b-v0.3-bnb-4bit: mistral_7b_finetuned
==((====))==  Unsloth 2025.9.4: Fast Mistral patching. Transformers: 4.56.1.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 5.799 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Unsloth: Tokenizing ["text"] (num_proc=16): 100%|██████████| 1308/1308 [00:00<00:00, 1780.46 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,308 | Num Epochs = 3 | Total steps = 492
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,7.900300
2,7.596300
3,8.069600
4,7.486000
5,7.806000
6,7.313800
7,8.645000
8,8.226000
9,7.190000
10,8.246600


TrainOutput(global_step=492, training_loss=1.929875117976491, metrics={'train_runtime': 1783.7809, 'train_samples_per_second': 2.2, 'train_steps_per_second': 0.276, 'total_flos': 2.156479376051405e+16, 'train_loss': 1.929875117976491, 'epoch': 3.0})

### Results

In [6]:
# Evaluate on the test dataset
test_accuracy, avg_processing_time = benchmark_model(model, tokenizer)
print(f"Model: {model_name}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Average Prompt Processing Time: {avg_processing_time:.4f} seconds")

# Save best model checkpoint
best_model_dir = output_dir
best_accuracy = test_accuracy
if os.path.exists("best_model_checkpoint"):
    shutil.rmtree("best_model_checkpoint")
shutil.copytree(output_dir, "best_model_checkpoint")


Map: 100%|██████████| 337/337 [00:00<00:00, 5683.43 examples/s]


Model: ./models/mistral-7b-v0.3-bnb-4bit
Test Accuracy: 99.70%
Average Prompt Processing Time: 0.7130 seconds


'best_model_checkpoint'